# Неделя 5. Метрики классификации и решение — семинар

## Преподавательская часть

Сегодня нас интересует не устройство нового алгоритма, а то, **что делать после `fit()`**:

$$
\text{данные}
\rightarrow
\text{модель}
\rightarrow
\hat p(x)
\rightarrow
\text{решение при пороге}
\rightarrow
\text{метрики}
\rightarrow
\text{ROC / PR}
\rightarrow
\text{качество вероятностей}.
$$

Будем двигаться маленькими шагами: **одна новая идея — одна небольшая ячейка**.
Сначала разберём инструменты sklearn по отдельности и только потом начнём собирать сравнительные таблицы.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import CalibrationDisplay, calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42


## 1. Данные: Blood Transfusion Service Center

Набор содержит сведения о **748 донорах** из центра переливания крови в Hsin-Chu City, Taiwan.

Признаки:

- `Recency` — сколько месяцев прошло с последней сдачи крови;
- `Frequency` — сколько раз человек сдавал кровь;
- `Monetary` — суммарный объём сданной крови;
- `Time` — сколько месяцев прошло с первой сдачи крови.

Целевая переменная показывает, сдавал ли человек кровь в рассматриваемом последующем периоде.

Источник: Yeh, I. (2008), **Blood Transfusion Service Center**, UCI Machine Learning Repository.  
DOI: `10.24432/C5GS39`, лицензия CC BY 4.0.


In [ ]:
data_path = Path("blood_transfusion.csv")
blood = pd.read_csv(data_path)

blood.head()


In [ ]:
blood.shape


In [ ]:
blood.isna().sum()


Положительным классом будем считать **`donated`**.

Сначала посмотрим, сколько объектов каждого класса есть в данных.


In [ ]:
blood["Class"].value_counts()


In [ ]:
blood["Class"].value_counts(normalize=True).round(3)


Положительных объектов около четверти.

Уже отсюда возникает первый вопрос:

> **Если модель показывает высокую Accuracy, достаточно ли этого?**


### `train_test_разбиение`

Вызов делит исходные данные на две части.

Здесь важны четыре аргумента:

- `X, y` — признаки и целевая переменная;
- `test_size=0.25` — во вторую часть попадёт 25% объектов;
- `stratify=y` — сохраняем примерно ту же долю классов в обеих частях;
- `random_state=42` — фиксируем случайное разбиение, чтобы результат воспроизводился.


In [ ]:
X = blood.drop(columns="Class")
y = (blood["Class"] == "donated").astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("train:", X_train.shape, "positive share =", round(y_train.mean(), 3))
print("validation:", X_val.shape, "positive share =", round(y_val.mean(), 3))


Вторую часть будем использовать как **валидационную выборку**.

Сегодня мы будем:

- считать на ней метрики;
- выбирать на ней рабочий порог.

Поэтому это ещё не независимая финальная оценка модели.
К разделению данных на обучающую, валидационную и тестовую выборки вернёмся отдельно.


## 2. Обучим несколько готовых классификаторов

Возьмём:

- `DummyClassifier(strategy="prior")` — простую базовую модель;
- `LogisticRegression` — уже знакомую модель;
- `KNeighborsClassifier` — пока просто второй классификатор с тем же интерфейсом sklearn.

Для Logistic Regression и kNN масштабируем признаки.
Масштабирование уже встречалось в предыдущих работах, поэтому здесь отдельно его не разбираем.


### `StandardScaler`

`fit_transform(X_train)` сначала оценивает среднее и стандартное отклонение **только по обучающей выборке**,
а затем масштабирует её.

`transform(X_val)` применяет уже найденные параметры к валидационной выборке.
Так информация из валидационной части не участвует в настройке преобразования.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)


### Параметры моделей

- `DummyClassifier(strategy="prior")` использует только доли классов в обучающей выборке;
- `LogisticRegression(max_iter=1000)` — увеличиваем допустимое число итераций оптимизации;
- `KNeighborsClassifier(n_neighbors=15)` — прогноз строится по 15 ближайшим соседям.

Сегодня устройство kNN не разбираем: здесь он нужен как ещё один классификатор с тем же интерфейсом sklearn.


In [ ]:
dummy = DummyClassifier(strategy="prior")
logreg = LogisticRegression(max_iter=1000)
knn = KNeighborsClassifier(n_neighbors=15)


In [ ]:
dummy.fit(X_train, y_train)
logreg.fit(X_train_scaled, y_train)
knn.fit(X_train_scaled, y_train)


### Почему `strategy="prior"`, а не `most_frequent`?

Для `predict()` обе стратегии выбирают самый частый класс.

Разница появляется в `predict_proba()`:

- `most_frequent` фактически выдаёт вероятность 1 самому частому классу;
- `prior` возвращает доли классов из обучающей выборки.

Поэтому `prior` удобнее как простая базовая модель, когда дальше мы будем сравнивать и вероятности.


## 3. Прогнозы классов: разбираем метрики по одной

Сначала работаем только с Logistic Regression и обычным `predict()`.


In [ ]:
y_pred_logreg = logreg.predict(X_val_scaled)

pd.DataFrame({
    "y_true": y_val.to_numpy()[:15],
    "y_pred": y_pred_logreg[:15],
})


### Accuracy

Напомним определение:

$$
Accuracy=\frac{\text{число правильных прогнозов}}{N}.
$$


`accuracy_score(y_true, y_pred)` получает:

- `y_true` — истинные классы;
- `y_pred` — предсказанные классы.

Результат — доля правильных прогнозов.


In [ ]:
accuracy_score(y_val, y_pred_logreg)


Получилось около 77%.

Само по себе число выглядит неплохо. Но сначала сравним его с простейшей базовой моделью.


In [ ]:
y_pred_dummy = dummy.predict(X_val)

accuracy_score(y_val, y_pred_dummy)


Dummy почти догоняет Logistic Regression по Accuracy.

Значит, нужно посмотреть **какие именно ошибки** делает модель.


### Матрица ошибок: сначала числа


`confusion_matrix(y_true, y_pred)` получает те же два массива классов
и возвращает таблицу числа объектов в четырёх исходах классификации.


In [ ]:
cm_logreg = confusion_matrix(y_val, y_pred_logreg)
cm_logreg


В `scikit-learn` строки — настоящий класс, столбцы — предсказанный:

$$
\begin{pmatrix}
TN & FP\\
FN & TP
\end{pmatrix}.
$$


In [ ]:
tn, fp, fn, tp = cm_logreg.ravel()
tn, fp, fn, tp


### Та же матрица — стандартной визуализацией sklearn

`ConfusionMatrixDisplay` — штатный sklearn-объект для визуализации confusion matrix.
Для обычной задачи не обязательно вручную собирать heatmap.


`ConfusionMatrixDisplay.from_predictions(...)` строит матрицу ошибок непосредственно по истинным и предсказанным классам.

Здесь:

- первые два аргумента — `y_true` и `y_pred`;
- `display_labels` задаёт подписи классов;
- `cmap="Blues"` выбирает цветовую шкалу;
- `values_format="d"` выводит целые количества объектов.


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_val,
    y_pred_logreg,
    display_labels=["not donated", "donated"],
    cmap="Blues",
    values_format="d",
)
plt.title("Logistic Regression, стандартное решение")
plt.show()


### Precision

Для положительного класса:

$$
Precision=\frac{TP}{TP+FP}.
$$

Вопрос: **какая доля объектов, которым мы предсказали класс 1, действительно относится к классу 1?**


`precision_score(y_true, y_pred)` получает истинные и предсказанные классы
и считает Precision для положительного класса.


In [ ]:
precision_score(y_val, y_pred_logreg)


### Recall

$$
Recall=\frac{TP}{TP+FN}.
$$

Вопрос: **какую долю всех настоящих объектов класса 1 мы нашли?**


`recall_score(y_true, y_pred)` получает истинные и предсказанные классы
и считает Recall для положительного класса.


In [ ]:
recall_score(y_val, y_pred_logreg)


Вот где обнаруживается проблема стандартного решения Logistic Regression:
на валидационной выборке 44 положительных объекта, а модель нашла только 4 из них.


### F1

$$
F_1=
\frac{2\cdot Precision\cdot Recall}
{Precision+Recall}.
$$


`f1_score(y_true, y_pred)` получает те же два массива
и считает гармоническое среднее Precision и Recall.


In [ ]:
f1_score(y_val, y_pred_logreg)


### `classification_report`

Когда отдельные метрики уже понятны, удобно собрать их в стандартную сводку.


`classification_report(y_true, y_pred, ...)` собирает несколько метрик в одну таблицу.

Дополнительные аргументы здесь:

- `target_names` — человекочитаемые названия классов;
- `zero_division=0` — если знаменатель какой-то метрики равен нулю, вернуть 0 вместо предупреждения;
- `digits=3` — показывать три знака после запятой.


In [ ]:
print(
    classification_report(
        y_val,
        y_pred_logreg,
        target_names=["not donated", "donated"],
        zero_division=0,
        digits=3,
    )
)


`classification_report` показывает Precision, Recall, F1 и `support` для каждого класса.

Сегодня нам достаточно строк двух классов и Accuracy.
`macro avg` и `weighted avg` системно понадобятся позже.


### Теперь сравним три модели

Только после того как каждую метрику разобрали отдельно, имеет смысл собрать компактную таблицу.

Здесь сравниваем **решения при стандартном `predict()`** для трёх моделей.


In [ ]:
models = {
    "Dummy prior": (dummy, X_val),
    "Logistic Regression": (logreg, X_val_scaled),
    "kNN (k=15)": (knn, X_val_scaled),
}

hard_predictions = {
    name: model.predict(X_eval)
    for name, (model, X_eval) in models.items()
}


In [ ]:
hard_rows = []

for name, y_pred in hard_predictions.items():
    hard_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1": f1_score(y_val, y_pred, zero_division=0),
    })

hard_metrics = pd.DataFrame(hard_rows).set_index("model")
hard_metrics.round(3)


Что видно:

- Dummy показывает, насколько мало сама Accuracy говорит при дисбалансе;
- при стандартном `predict()` kNN заметно лучше Logistic Regression по Recall и F1;
- но это ещё сравнение конкретных решений, которые модели получили при своих стандартных правилах.

Следующий вопрос: что произойдёт, если у одной и той же Logistic Regression изменить только порог?


## 4. От `predict()` к `predict_proba()`

`predict()` уже возвращает готовый класс.

Теперь посмотрим на вероятности, из которых Logistic Regression получает это решение.


In [ ]:
logreg.classes_


`classes_` показывает порядок столбцов в `predict_proba()`.

У нас классы `[0, 1]`, поэтому второй столбец — вероятность положительного класса.


`predict_proba(X)` получает признаки объектов и возвращает по одному столбцу вероятностей для каждого класса.

Порядок столбцов проверяем через `model.classes_`.


In [ ]:
proba_logreg = logreg.predict_proba(X_val_scaled)

pd.DataFrame(
    proba_logreg[:8],
    columns=["P(y=0)", "P(y=1)"],
).round(3)


Для каждого объекта две вероятности складываются в 1.
Нас дальше интересует только вероятность класса 1.


In [ ]:
p_logreg = proba_logreg[:, 1]

p_logreg[:10].round(3)


### Как `predict()` связан с порогом 0.5?

Для бинарной Logistic Regression стандартное решение эквивалентно правилу

$$
\hat y = \mathbb I[\hat p \ge 0.5].
$$


In [ ]:
y_pred_from_05 = (p_logreg >= 0.5).astype(int)

np.array_equal(y_pred_logreg, y_pred_from_05)


### Изменим только порог

Возьмём `threshold = 0.35`.

Выражение

```python
p_logreg >= threshold
```

сравнивает вероятность каждого объекта с порогом и возвращает `True` или `False`.

Затем

```python
.astype(int)
```

заменяет `True` на 1, а `False` на 0.

Так из вероятностей получаем новый прогноз класса — **без переобучения модели**.


In [ ]:
threshold = 0.35
y_pred_035 = (p_logreg >= threshold).astype(int)

pd.DataFrame({
    "p_hat": p_logreg[:12],
    "p_hat >= 0.35": p_logreg[:12] >= threshold,
    "y_pred": y_pred_035[:12],
}).round(3)


Модель не переобучалась.
Изменилось только правило, по которому вероятность превращается в класс.


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_val,
    y_pred_035,
    display_labels=["not donated", "donated"],
    cmap="Blues",
    values_format="d",
)
plt.title("Logistic Regression, threshold = 0.35")
plt.show()


Посчитаем знакомые метрики уже для нового решения.


In [ ]:
pd.Series({
    "selected_n": int(y_pred_035.sum()),
    "accuracy": accuracy_score(y_val, y_pred_035),
    "precision": precision_score(y_val, y_pred_035),
    "recall": recall_score(y_val, y_pred_035),
    "f1": f1_score(y_val, y_pred_035),
}).round(3)


При пороге 0.35 Logistic Regression выбирает 50 человек вместо 7 при стандартном 0.5.

Цена за рост Recall:

- Recall вырос примерно с 0.09 до 0.59;
- Precision немного снизилась;
- Accuracy почти не изменилась.

Это хороший пример того, почему **одна Accuracy не описывает рабочий режим классификатора**.


### Сравним изменение порога с изменением самой модели

Сопоставим:

- Logistic Regression при стандартном пороге 0.5;
- ту же Logistic Regression при 0.35;
- kNN при его стандартном `predict()`.


In [ ]:
y_pred_knn = knn.predict(X_val_scaled)

threshold_comparison = pd.DataFrame(
    {
        "LogReg, t=0.50": {
            "selected_n": int(y_pred_logreg.sum()),
            "accuracy": accuracy_score(y_val, y_pred_logreg),
            "precision": precision_score(y_val, y_pred_logreg),
            "recall": recall_score(y_val, y_pred_logreg),
            "f1": f1_score(y_val, y_pred_logreg),
        },
        "LogReg, t=0.35": {
            "selected_n": int(y_pred_035.sum()),
            "accuracy": accuracy_score(y_val, y_pred_035),
            "precision": precision_score(y_val, y_pred_035),
            "recall": recall_score(y_val, y_pred_035),
            "f1": f1_score(y_val, y_pred_035),
        },
        "kNN, predict()": {
            "selected_n": int(y_pred_knn.sum()),
            "accuracy": accuracy_score(y_val, y_pred_knn),
            "precision": precision_score(y_val, y_pred_knn),
            "recall": recall_score(y_val, y_pred_knn),
            "f1": f1_score(y_val, y_pred_knn),
        },
    }
).T

threshold_comparison.round(3)


Важно: изменение порога может повлиять на практические метрики не меньше,
чем переход к другой модели.

Поэтому **выбор модели** и **выбор порога для принятия решения** — разные задачи.


### Посмотрим ещё несколько порогов

Пока без универсальной функции — просто повторим уже понятные действия в цикле.


In [ ]:
rows = []

for threshold in [0.20, 0.35, 0.50, 0.60]:
    y_pred_t = (p_logreg >= threshold).astype(int)

    rows.append({
        "threshold": threshold,
        "selected_n": int(y_pred_t.sum()),
        "selected_share": y_pred_t.mean(),
        "precision": precision_score(y_val, y_pred_t, zero_division=0),
        "recall": recall_score(y_val, y_pred_t, zero_division=0),
        "f1": f1_score(y_val, y_pred_t, zero_division=0),
    })

four_thresholds = pd.DataFrame(rows)
four_thresholds.round(3)


`selected_n` — сколько объектов получили прогноз класса 1.

`selected_share` — какая доля всего списка получила прогноз класса 1:

$$
selected\_share
=
\frac{selected\_n}{N}.
$$

Поскольку `y_pred_t` состоит только из нулей и единиц, его среднее как раз равно этой доле.


Что видно из таблицы:

- при `t=0.20` мы выбираем больше половины списка и находим около 84% настоящих доноров, но Precision падает примерно до 0.39;
- при `t=0.50` выбираем всего 7 человек и почти не допускаем лишних приглашений, но Recall падает примерно до 0.09;
- `t=0.35` находится между этими крайностями: выбираем 50 человек, Recall около 0.59, Precision около 0.52.

Какой из этих режимов нужен — определяется **не самой моделью, а прикладной задачей**.


## 5. Выбираем порог под конкретное ограничение

Представим, что перед новой кампанией у центра есть список из **187 кандидатов** —
столько объектов сейчас в валидационной выборке.

Сотрудники успевают связаться **не более чем с 50 людьми**.

Цель:

> среди решений, которые выбирают не больше 50 кандидатов,
> найти такое, которое охватывает как можно больше тех, кто действительно сдаст кровь.

То есть:

- ограничение: `selected_n <= 50`;
- основная цель: максимальный Recall.


Для одного порога мы уже умеем считать всё вручную.
Теперь расчёт нужно повторять много раз — здесь helper действительно начинает экономить код.


In [ ]:
def metrics_at_threshold(y_true, y_score, threshold):
    """
    Посчитать hard-classification метрики при фиксированном пороге.

    Parameters
    ----------
    y_true : array-like of shape (n_samples,)
        Истинные бинарные метки классов: 0 или 1.

    y_score : array-like of shape (n_samples,)
        Score или вероятность положительного класса для тех же объектов.
        Здесь это `predict_proba(... )[:, 1]`.

    threshold : float
        Порог принятия решения.
        Объект получает прогноз 1, если `y_score >= threshold`,
        иначе получает прогноз 0.

    Returns
    -------
    dict
        Одна строка для будущей таблицы рабочих режимов:

        - `threshold` — использованный порог;
        - `selected_n` — число положительных прогнозов;
        - `selected_share` — доля положительных прогнозов среди всех объектов;
        - `accuracy` — Accuracy;
        - `precision` — Precision положительного класса;
        - `recall` — Recall положительного класса;
        - `f1` — F1 положительного класса.
    """
    y_pred = (np.asarray(y_score) >= threshold).astype(int)

    return {
        "threshold": float(threshold),
        "selected_n": int(y_pred.sum()),
        "selected_share": float(y_pred.mean()),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


Сначала проверим helper на знакомом пороге 0.5.


In [ ]:
pd.Series(
    metrics_at_threshold(y_val, p_logreg, 0.5)
).round(3)


### Какие пороги вообще нужно проверять?

На конечной выборке прогноз класса меняется только тогда,
когда порог пересекает одно из наблюдаемых значений вероятности.

Поэтому достаточно проверить уникальные значения `p_logreg`.


In [ ]:
candidate_thresholds = np.sort(np.unique(p_logreg))[::-1]

print("Число объектов:", len(p_logreg))
print("Число уникальных score:", len(candidate_thresholds))
candidate_thresholds[:10]


Теперь строим таблицу:

> **одна строка = один возможный режим работы Logistic Regression на этой валидационной выборке.**


In [ ]:
threshold_table = pd.DataFrame([
    metrics_at_threshold(y_val, p_logreg, threshold)
    for threshold in candidate_thresholds
])

threshold_table.head(10).round(3)


Сначала применяем прикладное ограничение: сотрудники могут связаться максимум с 50 людьми.


In [ ]:
feasible = threshold_table[
    threshold_table["selected_n"] <= 50
]

feasible.shape


Теперь среди допустимых режимов максимизируем Recall.

Если Recall совпадает, дополнительно сравниваем Precision:
из двух решений с одинаковым охватом настоящих доноров выбираем то,
где среди приглашённых выше доля тех, кто действительно сдаст кровь.


In [ ]:
feasible_ranked = feasible.sort_values(
    ["recall", "precision"],
    ascending=[False, False],
)

preview = feasible_ranked.head(10).copy()

preview.round({
    "threshold": 6,
    "selected_share": 3,
    "accuracy": 3,
    "precision": 3,
    "recall": 3,
    "f1": 3,
})


Обратите внимание на близкие значения порога.

Например, первые две строки имеют пороги примерно `0.355748` и `0.356117`.
Если округлить их до трёх знаков, оба выглядят как `0.356`,
хотя это **разные пороги** и поэтому они могут давать разные прогнозы и метрики.

Именно поэтому в этой таблице для `threshold` оставляем больше знаков.


In [ ]:
best = feasible_ranked.iloc[0]
best.round(3)


Получился порог около 0.356.

Он выбирает 50 кандидатов и находит примерно 59% будущих доноров.

Это **не лучший порог вообще**.
Он лучший для текущего ограничения «не больше 50 контактов»
и цели «охватить как можно больше будущих доноров».


In [ ]:
chosen_threshold = float(best["threshold"])
y_pred_chosen = (p_logreg >= chosen_threshold).astype(int)

ConfusionMatrixDisplay.from_predictions(
    y_val,
    y_pred_chosen,
    display_labels=["not donated", "donated"],
    cmap="Blues",
    values_format="d",
)
plt.title(f"Logistic Regression, chosen threshold = {chosen_threshold:.3f}")
plt.show()


На этом конкретная задача кампании решена:
мы выбрали **одну рабочую точку** — порог, который удовлетворяет ограничению на число контактов.

Но такой вывод привязан именно к сегодняшнему ограничению в 50 человек.

Если завтра лимит изменится — изменится и выбранный порог.
А если мы хотим сравнить две модели **до** выбора конкретного ограничения,
полезно посмотреть, насколько хорошо каждая из них ранжирует объекты при всех возможных порогах.

Для этого и нужны ROC- и PR-кривые.


## 6. ROC: качество ранжирования по всем порогам

Напомним идеи лекции.

$$
TPR = Recall = \frac{TP}{TP+FN},
$$

$$
FPR = \frac{FP}{FP+TN}.
$$

Каждый порог задаёт свой прогноз классов, свою матрицу ошибок
и одну точку

$$
(FPR,\ TPR).
$$

Меняя порог, получаем ROC-кривую.


### Функция `roc_curve`

Вызов:

```python
roc_curve(y_true, y_score)
```

- `y_true` — истинные метки 0/1;
- `y_score` — непрерывная оценка положительного класса; у нас это `p_logreg`.

Функция возвращает три массива:

- FPR;
- TPR;
- пороги, которым соответствуют эти точки.


In [ ]:
fpr_logreg, tpr_logreg, roc_thresholds = roc_curve(
    y_val,
    p_logreg,
)

pd.DataFrame({
    "threshold": roc_thresholds[:8],
    "FPR": fpr_logreg[:8],
    "TPR": tpr_logreg[:8],
}).round(3)


Для построения графика в sklearn есть `RocCurveDisplay`.

Сначала построим ROC **без выделенной рабочей точки**.


`RocCurveDisplay.from_predictions(y_true, y_score, ...)` строит ROC по готовым оценкам модели.

Здесь:

- `y_true` — истинные классы;
- `y_score` — вероятности положительного класса;
- `name` — подпись линии;
- `plot_chance_level=True` — добавить диагональ случайного ранжирования.


In [ ]:
RocCurveDisplay.from_predictions(
    y_val,
    p_logreg,
    name="Logistic Regression",
    plot_chance_level=True,
)
plt.title("ROC: Logistic Regression")
plt.show()


Как читать ROC:

- по горизонтали — FPR;
- по вертикали — TPR;
- левый верхний угол соответствует большому TPR при маленьком FPR;
- диагональ показывает уровень случайного ранжирования;
- каждая точка кривой соответствует некоторому порогу.

ROC показывает, **какие сочетания TPR и FPR доступны модели**,
но сама не выбирает рабочий порог.


### Где на ROC находится выбранный нами порог?

Сначала получим его координаты из матрицы ошибок выбранного решения.


In [ ]:
tn, fp, fn, tp = confusion_matrix(
    y_val,
    y_pred_chosen,
).ravel()

chosen_fpr = fp / (fp + tn)
chosen_tpr = tp / (tp + fn)

pd.Series({
    "FPR": chosen_fpr,
    "TPR / Recall": chosen_tpr,
}).round(3)


Теперь просто нанесём эту точку поверх уже знакомой ROC-кривой.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

RocCurveDisplay.from_predictions(
    y_val,
    p_logreg,
    name="Logistic Regression",
    plot_chance_level=True,
    ax=ax,
)

ax.scatter(
    chosen_fpr,
    chosen_tpr,
    s=90,
    zorder=5,
    label=f"chosen t = {chosen_threshold:.3f}",
)
ax.legend()
plt.show()


Выделенная точка — решение, которое мы выбрали для ограничения в 50 контактов.

Остальная ROC-кривая показывает, какие другие сочетания TPR и FPR
получились бы при других порогах той же модели.


### Сравним ранжирование трёх моделей

Для ROC нужны не готовые классы, а непрерывные оценки или вероятности положительного класса.


In [ ]:
probabilities = {
    "Dummy prior": dummy.predict_proba(X_val)[:, 1],
    "Logistic Regression": p_logreg,
    "kNN (k=15)": knn.predict_proba(X_val_scaled)[:, 1],
}


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for name, y_score in probabilities.items():
    RocCurveDisplay.from_predictions(
        y_val,
        y_score,
        name=name,
        ax=ax,
    )

ax.plot([0, 1], [0, 1], "--", label="chance level")
ax.set_title("ROC: сравнение моделей")
ax.legend()
plt.show()


Dummy даёт уровень случайного ранжирования.

Logistic Regression и kNN заметно лучше этой базовой модели и близки друг к другу.


### Почему ROC у kNN состоит из более крупных шагов?

`KNeighborsClassifier(n_neighbors=15)` оценивает вероятность
как долю объектов положительного класса среди 15 соседей.

Поэтому возможны только значения вида

$$
0,\ \frac1{15},\ \frac2{15},\ldots,\frac{15}{15}.
$$

Проверим, сколько разных значений вероятности получилось у двух моделей.


In [ ]:
pd.Series({
    "Logistic Regression": np.unique(p_logreg).size,
    "kNN (k=15)": np.unique(probabilities["kNN (k=15)"]).size,
}, name="unique score values")


У Logistic Regression вероятности почти непрерывны, поэтому порог может пересечь много разных значений.

У kNN значения дискретны — здесь всего несколько уровней.
Значит, у kNN меньше разных рабочих точек, а шаги ROC крупнее.

Форма линии здесь связана с набором возможных значений вероятности
и сама по себе ничего не говорит о том, какая модель лучше.


### ROC-AUC: одно число для качества ранжирования

В лекции мы интерпретировали ROC-AUC так:

> если случайно взять один положительный и один отрицательный объект,
> хорошая модель должна дать положительному объекту более высокую оценку.

ROC-AUC сворачивает качество такого ранжирования в одно число.
Она **не оценивает один выбранный порог**.


`roc_auc_score(y_true, y_score)` получает истинные классы и непрерывные оценки модели
и возвращает площадь под ROC-кривой.


In [ ]:
roc_auc_values = pd.Series({
    name: roc_auc_score(y_val, y_score)
    for name, y_score in probabilities.items()
}, name="ROC-AUC")

roc_auc_values.round(3)


На этой выборке Logistic Regression немного выше kNN по ROC-AUC.

Это не противоречит тому, что kNN раньше выглядел лучше по Recall при стандартном `predict()`:

- Recall и F1 относятся к одному конкретному решению;
- ROC-AUC оценивает порядок объектов независимо от одного фиксированного порога.

Мы задаём модели разные вопросы — поэтому и ответы могут различаться.


## 7. Precision–Recall: другой взгляд на те же пороги

ROC показывает соотношение TPR и FPR.

Но иногда важнее другой вопрос:

> **если модель выбрала некоторую группу как положительную, какая доля этой группы действительно положительна и какую долю всех положительных объектов мы нашли?**

Для этого смотрим на

$$
Precision = \frac{TP}{TP+FP},
$$

$$
Recall = \frac{TP}{TP+FN}.
$$

Каждый порог снова задаёт одну точку

$$
(Recall,\ Precision).
$$


### Функция `precision_recall_curve`

Вызов:

```python
precision_recall_curve(y_true, y_score)
```

- `y_true` — истинные метки;
- `y_score` — непрерывная оценка положительного класса.

Функция возвращает:

- Precision;
- Recall;
- пороги.


In [ ]:
precision_logreg, recall_logreg, pr_thresholds = precision_recall_curve(
    y_val,
    p_logreg,
)

print("len(precision) =", len(precision_logreg))
print("len(recall)    =", len(recall_logreg))
print("len(thresholds)=", len(pr_thresholds))


Почему `precision` и `recall` на один элемент длиннее массива порогов?

sklearn добавляет в конец PR-кривой служебную точку

`Recall = 0, Precision = 1`.

Она нужна только для завершения графика и не соответствует отдельному порогу.
Посмотрим на последние строки явно.


In [ ]:
pd.DataFrame({
    "threshold": [
        pr_thresholds[-2],
        pr_thresholds[-1],
        np.nan,
    ],
    "precision": precision_logreg[-3:],
    "recall": recall_logreg[-3:],
}, index=["предпоследний threshold", "последний threshold", "служебная конечная точка"]).round(3)


Для готового графика используем `PrecisionRecallDisplay`.

Сначала снова **без выбранной рабочей точки**.


`PrecisionRecallDisplay.from_predictions(y_true, y_score, ...)` строит PR-кривую по готовым оценкам модели.

Аргументы устроены так же:

- `y_true` — истинные классы;
- `y_score` — вероятность положительного класса;
- `name` — подпись линии;
- `plot_chance_level=True` — показать уровень, соответствующий доле положительного класса.


In [ ]:
PrecisionRecallDisplay.from_predictions(
    y_val,
    p_logreg,
    name="Logistic Regression",
    plot_chance_level=True,
)
plt.title("Precision–Recall: Logistic Regression")
plt.show()


Как читать PR-кривую:

- по горизонтали — Recall;
- по вертикали — Precision;
- при движении вправо мы находим всё большую долю положительных объектов;
- при этом важно, какая доля выбранной группы действительно положительна;
- горизонтальный уровень базовой модели примерно совпадает с долей положительного класса.

PR-кривая особенно полезна, когда нас интересует **состав выбранной положительной группы**
и положительный класс встречается сравнительно редко.


### Нанесём выбранный порог

Для нашего решения уже известны Precision и Recall.
Именно они задают его точку на PR-кривой.


In [ ]:
chosen_precision = precision_score(y_val, y_pred_chosen)
chosen_recall = recall_score(y_val, y_pred_chosen)

pd.Series({
    "Precision": chosen_precision,
    "Recall": chosen_recall,
}).round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

PrecisionRecallDisplay.from_predictions(
    y_val,
    p_logreg,
    name="Logistic Regression",
    plot_chance_level=True,
    ax=ax,
)

ax.scatter(
    chosen_recall,
    chosen_precision,
    s=90,
    zorder=5,
    label=f"chosen t = {chosen_threshold:.3f}",
)
ax.legend()
plt.show()


### Сравним модели по PR


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for name, y_score in probabilities.items():
    PrecisionRecallDisplay.from_predictions(
        y_val,
        y_score,
        name=name,
        ax=ax,
    )

ax.axhline(
    y_val.mean(),
    linestyle="--",
    label=f"positive prevalence = {y_val.mean():.3f}",
)
ax.set_title("Precision–Recall: сравнение моделей")
ax.legend()
plt.show()


Что видно:

- Dummy находится примерно на уровне доли положительного класса;
- Logistic Regression и kNN заметно лучше базовой модели;
- Logistic Regression в среднем немного лучше ранжирует объекты положительного класса;
- у kNN меньше разных значений вероятности, поэтому PR-кривая состоит из более крупных шагов.


### Average Precision

AP сворачивает PR-кривую в одно число.
В лекции мы получили её как сумму площадей ступенчатой конструкции PR.


`average_precision_score(y_true, y_score)` получает истинные классы и оценки модели
и возвращает Average Precision.


In [ ]:
ap_values = pd.Series({
    name: average_precision_score(y_val, y_score)
    for name, y_score in probabilities.items()
}, name="Average Precision")

ap_values.round(3)


На этой выборке:

- у базовой модели AP примерно равна доле положительного класса;
- у Logistic Regression AP около 0.54;
- у kNN — около 0.51.

Обе модели ранжируют объекты заметно лучше случайного порядка,
а Logistic Regression здесь немного лучше именно по PR-критерию.


### Как выбирать между пороговыми метриками, ROC и PR?

- Если уже выбран конкретный порог — смотрим матрицу ошибок, Precision, Recall, F1 и другие метрики этого решения.
- Если хотим сравнить **ранжирование** моделей без привязки к одному порогу — смотрим ROC/ROC-AUC и PR/AP.
- Если особенно важен состав выбранной положительной группы или положительный класс редок — PR и AP обычно дают более непосредственную картину.

ROC и PR не выбирают решение за нас.
Они показывают, **какие рабочие режимы вообще доступны благодаря ранжированию модели**.


Теперь остаётся ещё один вопрос.

Хорошее ранжирование ещё не означает, что число `0.7`
можно понимать как хорошо оценённую вероятность около 70%.

Для этого отдельно проверим шкалу вероятностей.


## 8. Калибровка и качество вероятностного прогноза


### Сначала `calibration_curve`

Вызов:

```python
calibration_curve(y_true, y_prob, n_группы=5, strategy="quantile")
```

- `y_true` — истинные метки 0/1;
- `y_prob` — прогнозы вероятности класса 1;
- `n_группы=5` — число групп;
- `strategy="quantile"` — группы формируются так, чтобы в них было примерно одинаковое число объектов.

Альтернатива `strategy="uniform"` делит сам интервал вероятностей от 0 до 1
на равные по ширине части.

На нашей небольшой выборке вариант `quantile` удобен тем,
что уменьшает риск почти пустых групп.


In [ ]:
fraction_positive, mean_predicted = calibration_curve(
    y_val,
    p_logreg,
    n_bins=5,
    strategy="quantile",
)

calibration_logreg = pd.DataFrame({
    "mean predicted probability": mean_predicted,
    "observed positive rate": fraction_positive,
})

calibration_logreg.round(3)


Для каждой группы сравниваем:

- средний прогноз $\bar p_k$;
- наблюдаемую долю положительного класса $\bar y_k$.

Если шкала хорошо согласуется с частотами, точки должны быть около

$$
y=x.
$$


Теперь построим reliability diagram для одной Logistic Regression.


`CalibrationDisplay.from_predictions(y_true, y_prob, ...)` строит reliability diagram.

Здесь:

- `y_true` — истинные классы;
- `y_prob` — прогнозы вероятности класса 1;
- `n_группы=5` — число групп;
- `strategy="quantile"` — группы примерно одинаковы по числу объектов;
- `name` — подпись модели.


In [ ]:
CalibrationDisplay.from_predictions(
    y_val,
    p_logreg,
    n_bins=5,
    strategy="quantile",
    name="Logistic Regression",
)
plt.title("Reliability diagram: Logistic Regression")
plt.show()


На этой небольшой выборке нет одного простого систематического смещения:

- в некоторых группах Logistic Regression немного завышает вероятность;
- в верхней группе, наоборот, средний прогноз около 0.46,
  а наблюдаемая доля положительного класса около 0.55.

Всего в валидационной выборке 187 объектов, поэтому график заметно шумит.
Здесь он нужен прежде всего для понимания того,
**в каких диапазонах прогнозы расходятся с наблюдаемыми частотами**.


### Сравним шкалы вероятностей моделей


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for name, y_score in probabilities.items():
    CalibrationDisplay.from_predictions(
        y_val,
        y_score,
        n_bins=5,
        strategy="quantile",
        name=name,
        ax=ax,
    )

ax.set_title("Reliability diagram: сравнение моделей")
plt.show()


На этой валидационной выборке точки kNN визуально часто ближе к диагонали,
чем точки Logistic Regression.

Но одного такого графика недостаточно для уверенного вывода:

- в каждой точке уже усреднена группа объектов;
- результат зависит от разбиения на группы;
- выборка небольшая, поэтому наблюдаемые частоты шумят.

Reliability diagram отвечает на вопрос **«где шкала вероятностей расходится с частотами?»**.
Теперь посмотрим на численные оценки вероятностных прогнозов по отдельным объектам.


### Log loss

Для одного объекта:

$$
L(y,\hat p)
=
-y\log\hat p-(1-y)\log(1-\hat p).
$$

Чем меньше средний Log loss, тем лучше.
Особенность этой функции: уверенный неправильный прогноз получает очень большой штраф.


`log_loss(y_true, y_prob)` получает истинные классы и прогнозы вероятности положительного класса.
Возвращается средний Log loss по выборке.


In [ ]:
logloss_values = pd.Series({
    name: log_loss(y_val, y_score)
    for name, y_score in probabilities.items()
}, name="Log loss")

logloss_values.round(3)


Здесь возникает неожиданный результат:
kNN по Log loss оказался даже хуже простой базовой модели.

Это не ошибка вычисления.
Посмотрим, какие вероятности kNN присвоил положительным объектам.


In [ ]:
knn_scores = probabilities["kNN (k=15)"]
y_val_array = y_val.to_numpy()

knn_probability_table = pd.DataFrame({
    "y_true": y_val_array,
    "p_knn": knn_scores,
})

knn_probability_table[
    knn_probability_table["y_true"] == 1
].sort_values("p_knn").head()


Есть настоящий объект положительного класса, которому kNN присвоил вероятность `0.0`.

Для kNN это возможно естественным образом:
среди 15 соседей не оказалось ни одного объекта положительного класса.

Для Log loss такой уверенный промах очень дорог.


In [ ]:
eps = np.finfo(float).eps
p_knn_clipped = np.clip(knn_scores, eps, 1 - eps)

knn_logloss_per_object = -(
    y_val_array * np.log(p_knn_clipped)
    + (1 - y_val_array) * np.log(1 - p_knn_clipped)
)

worst_idx = np.argmax(knn_logloss_per_object)

pd.Series({
    "y_true": y_val_array[worst_idx],
    "p_knn": knn_scores[worst_idx],
    "object_log_loss": knn_logloss_per_object[worst_idx],
    "contribution_to_mean_log_loss": (
        knn_logloss_per_object[worst_idx] / len(y_val_array)
    ),
}).round(3)


Один такой объект добавляет к **среднему** Log loss около 0.19.

Поэтому reliability diagram может выглядеть вполне прилично по группа averages,
а Log loss одновременно сигнализирует о редком, но очень уверенном провале.
Они отвечают на разные диагностические вопросы.


### Brier score

Для бинарной задачи:

$$
Brier
=
\frac1N
\sum_{i=1}^N
(y_i-\hat p_i)^2.
$$

Это средняя квадратичная ошибка вероятностного прогноза.
Чем меньше Brier score, тем лучше.

В отличие от Log loss, Brier не увеличивает штраф без границы
при вероятности, близкой к 0 или 1.


`brier_score_loss(y_true, y_prob)` получает те же два массива
и возвращает средний Brier score.


In [ ]:
brier_values = pd.Series({
    name: brier_score_loss(y_val, y_score)
    for name, y_score in probabilities.items()
}, name="Brier")

brier_values.round(3)


По Brier kNN и Logistic Regression почти равны, причём kNN здесь чуть лучше.

Почему это совместимо с плохим Log loss?

- при `y=1` и `p=0` вклад в Brier равен 1;
- тот же уверенный промах даёт очень большой вклад в Log loss;
- на остальных объектах kNN предсказывает достаточно неплохо.

Поэтому различие не означает, что одна из метрик «сломалась».
Они по-разному штрафуют уверенные ошибки.


### Что делать, если Log loss и Brier дают разный порядок моделей?

Не выбирать метрику после просмотра результата.

Сначала решаем, какие ошибки вероятностей для задачи особенно нежелательны:

- Log loss очень сильно штрафует уверенные неправильные прогнозы;
- Brier использует квадратичный штраф и реагирует на такие ошибки мягче;
- reliability diagram показывает, в каких диапазонах вероятностей прогнозы расходятся с наблюдаемыми частотами.

По одному Log loss или Brier нельзя понять,
**где именно** шкала вероятностей завышена или занижена.
Для этого и нужен reliability diagram.


## 9. Короткое резюме

Сегодня один и тот же результат классификатора рассматривали с трёх сторон:

| Вопрос | Что используем |
|---|---|
| Как работает конкретное решение при выбранном пороге? | матрица ошибок, Accuracy, Precision, Recall, F1 |
| Насколько хорошо модель ранжирует объекты при разных порогах? | ROC/ROC-AUC, PR/AP |
| Насколько хороши сами прогнозируемые вероятности? | reliability diagram, Log loss, Brier |

Практический порядок работы простой:

1. сначала формулируем задачу и ограничения;
2. затем выбираем подходящую метрику или кривую;
3. только после этого выбираем модель и, если нужно, рабочий порог.

В самостоятельной части применим эту схему к нескольким другим задачам.


---

# Часть 2. Самостоятельные кейсы

Теперь меняем режим работы: модели уже обучены, а признаки подготовлены.
Ваша задача — по постановке понять, **какое качество решения действительно важно**, выбрать подходящую метрику или порог и содержательно объяснить результат.


Ниже — несколько независимых задач из разных предметных областей.

За оставшееся время семинара решите столько кейсов, сколько успеете.
Ориентир — **2–3 кейса**.

Один из решённых кейсов будьте готовы объяснить подробно:

1. что здесь является положительным классом;
2. какое ограничение или цель заданы;
3. почему вы используете именно эту метрику;
4. какой порог или модель вы выбрали;
5. чем пришлось пожертвовать.

По остальным кейсам достаточно показать расчёты и короткий вывод.

Если успеваете решить три задачи, полезно взять их из разных разделов:
одну про выбор порога, одну про ROC/PR и одну про качество вероятностей.
Это рекомендация, а не обязательное правило.

После семинара рекомендуется самостоятельно пройти оставшиеся кейсы:
они специально построены по-разному, чтобы несколько раз потренировать
переход **от постановки задачи к выбору метрики и решения**.


## Что уже подготовлено

В каждом кейсе модель уже обучена.
Вам выдаётся небольшая таблица с истинным классом и прогнозами на отдельной выборке.

Обычно в ней есть столбцы:

- `y_true` — истинный класс;
- `score` — вероятность или непрерывная оценка положительного класса;
- `score_a`, `score_b` — оценки двух моделей, если их нужно сравнить.

Повторно обучать модели и готовить признаки не нужно:
сегодняшняя задача — **оценить прогнозы и принять решение**.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

DATA_DIR = Path("case_data")


## Общая функция: метрики при разных порогах

Эту функцию не нужно писать заново в каждом кейсе.

Она перебирает значения порога и для каждого решения сохраняет:

- число выбранных объектов;
- матрицу ошибок;
- Accuracy;
- Precision;
- Recall;
- Specificity;
- FPR;
- F1.

Напомним:

$$
Specificity = \frac{TN}{TN+FP},
\qquad
FPR = \frac{FP}{FP+TN}=1-Specificity.
$$


In [ ]:
def threshold_metrics_table(y_true, y_score):
    """
    Построить таблицу метрик для всех различных пороговых решений.

    Parameters
    ----------
    y_true : array-like of shape (n_samples,)
        Истинные бинарные метки: 0 или 1.

    y_score : array-like of shape (n_samples,)
        Вероятность или непрерывная оценка положительного класса
        для тех же объектов.

    Returns
    -------
    pandas.DataFrame
        Одна строка соответствует одному порогу.

        Столбцы:
        - threshold      — порог;
        - selected_n     — число прогнозов класса 1;
        - selected_share — доля прогнозов класса 1;
        - TN, FP, FN, TP — элементы матрицы ошибок;
        - accuracy       — Accuracy;
        - precision      — Precision;
        - recall         — Recall / TPR;
        - specificity    — TN / (TN + FP);
        - FPR            — FP / (FP + TN);
        - f1             — F1.
    """
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)

    thresholds = np.r_[
        np.inf,
        np.sort(np.unique(y_score))[::-1],
        -np.inf,
    ]

    rows = []

    for threshold in thresholds:
        y_pred = (y_score >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()

        specificity = tn / (tn + fp) if (tn + fp) else np.nan
        fpr = fp / (fp + tn) if (fp + tn) else np.nan

        rows.append({
            "threshold": threshold,
            "selected_n": int(y_pred.sum()),
            "selected_share": float(y_pred.mean()),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(
                y_true, y_pred, zero_division=0
            ),
            "recall": recall_score(
                y_true, y_pred, zero_division=0
            ),
            "specificity": specificity,
            "FPR": fpr,
            "f1": f1_score(
                y_true, y_pred, zero_division=0
            ),
        })

    return pd.DataFrame(rows)


## Ещё одна заготовка: верхние `k` объектов

В некоторых задачах ограничение задаётся не порогом,
а конкретным числом объектов, которые можно проверить, обзвонить или выбрать.

Тогда удобно отсортировать объекты по убыванию оценки модели
и взять первые `k`.


In [ ]:
def top_k_metrics(y_true, y_score, k):
    """
    Оценить качество списка из k объектов с максимальной оценкой модели.

    Parameters
    ----------
    y_true : array-like of shape (n_samples,)
        Истинные бинарные метки.

    y_score : array-like of shape (n_samples,)
        Оценка положительного класса.

    k : int
        Сколько объектов можно выбрать.

    Returns
    -------
    dict
        Число найденных положительных объектов, Precision,
        Recall и минимальная оценка среди выбранных объектов.
    """
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)

    order = np.argsort(y_score)[::-1]
    selected = order[:k]

    tp = int(y_true[selected].sum())
    positives = int(y_true.sum())

    return {
        "selected_n": len(selected),
        "TP": tp,
        "precision": tp / len(selected),
        "recall": tp / positives,
        "lowest_selected_score": y_score[selected].min(),
    }


---

# A. Выбор порога


## A1. Диагностика рака молочной железы

**Положительный класс:** злокачественная опухоль.

Модель используется как дополнительный предварительный инструмент: она помогает выделить случаи, которые требуют особенно внимательной дальнейшей проверки.

Здесь два типа ошибок имеют разный смысл:

- **FN:** злокачественный случай получил отрицательный прогноз и может не попасть в группу повышенного внимания;
- **FP:** доброкачественный случай модель пометила как подозрительный, поэтому потребуется лишняя дополнительная проверка.

В этой постановке сначала важно почти не пропускать злокачественные случаи. Поэтому предметное требование переводим в ограничение

$$
Recall \ge 0.98.
$$

Но можно получить высокий Recall, просто пометив подозрительными очень много случаев. Поэтому среди порогов, уже удовлетворяющих ограничению по Recall, выберите решение с **максимальной Precision**.

### Что нужно показать

- выбранный порог;
- Precision и Recall;
- матрицу ошибок;
- короткое объяснение: какой тип ошибки мы старались почти исключить и почему Accuracy здесь не является главным критерием.

Источник данных: Breast Cancer Wisconsin (Diagnostic), UCI, DOI `10.24432/C5DW2B`.

In [ ]:
a1 = pd.read_csv(DATA_DIR / "A1_breast_cancer.csv")

a1.head()


In [ ]:
y_true = a1["y_true"].to_numpy()
score = a1["score"].to_numpy()

table = threshold_metrics_table(y_true, score)

# TODO:
# 1. Оставьте строки с Recall >= 0.98.
# 2. Среди них выберите максимальную Precision.
# 3. Если несколько строк совпадают по Precision,
#    выберите более высокий порог.

# YOUR CODE HERE


In [ ]:
# TODO: постройте матрицу ошибок для выбранного порога.

# YOUR CODE HERE


**Вывод A1:**  
Напишите 2–4 предложения: какой порог выбран, что получилось с Precision/Recall
и какой тип ошибки мы сознательно старались почти исключить.


## A2. Bank Marketing: ограниченный ресурс call-центра

**Положительный класс:** клиент оформит срочный депозит.

Банк планирует телефонную маркетинговую кампанию. Для каждого клиента модель оценивает, насколько перспективным может оказаться звонок, а затем по прогнозу решается, кому сотрудники позвонят.

Прогноз нужен **до звонка**. Поэтому из исходных признаков заранее исключена длительность разговора: она становится известна только после звонка и не может помогать выбирать клиента.

В отдельной выборке находятся **1000 клиентов**, но сотрудники успевают позвонить **не более чем 150 людям**. Поэтому ресурсное ограничение превращается в формальное условие: положительный прогноз должны получить не более 150 клиентов.

Среди допустимых решений хочется найти как можно большую долю всех клиентов, которые действительно оформили бы депозит. Поэтому среди порогов, удовлетворяющих ограничению на число звонков, выбираем максимальный **Recall**.

### Задача

Выберите порог так, чтобы:

- положительный прогноз получили не более 150 клиентов;
- среди допустимых решений Recall был максимальным.

Затем повторите расчёт, если ресурсов хватает только на **100 звонков**.

### Что нужно показать

- оба выбранных порога;
- число выбранных клиентов;
- Precision и Recall для 150 и 100 звонков;
- короткое объяснение: почему изменение ресурса call-центра меняет рабочий порог и что происходит с Recall.

Источник данных: UCI Bank Marketing, DOI `10.24432/C5K306`.

In [ ]:
a2 = pd.read_csv(DATA_DIR / "A2_bank_marketing.csv")

y_true = a2["y_true"].to_numpy()
score = a2["score"].to_numpy()

table = threshold_metrics_table(y_true, score)

# TODO: решение для лимита 150 клиентов.
# YOUR CODE HERE


In [ ]:
# TODO: решение для лимита 100 клиентов.
# YOUR CODE HERE


**Вывод A2:**  
Сравните два режима и объясните,
как ограничение на число звонков повлияло на Precision и Recall.


## A3. Фильтр спама

**Положительный класс:** spam.

После прогноза система принимает реальное действие: письма с положительным решением автоматически отправляются в папку spam.

Ошибки означают следующее:

- **FN:** spam остался во входящих;
- **FP:** обычное письмо было ошибочно отправлено в spam.

В этой постановке ложноположительная ошибка особенно неприятна: среди скрытых сообщений может оказаться важное рабочее или личное письмо. Поэтому администратор сначала задаёт допустимую частоту таких ошибок:

$$
FPR \le 0.02.
$$

То есть ошибочно отправляться в spam должны не более 2% обычных писем.

Но внутри этого ограничения хочется остановить как можно больше настоящего spam. Поэтому среди допустимых порогов выберите решение с максимальным **Recall**.

### Затем

Повторите выбор при менее строгом ограничении

$$
FPR \le 0.05.
$$

### Что нужно показать

- оба порога;
- Recall и Precision;
- матрицу ошибок хотя бы для одного режима;
- объяснение: что система получила в обмен на разрешение чаще ошибочно блокировать обычные письма.

Источник данных: UCI Spambase, DOI `10.24432/C53G6X`.

In [ ]:
a3 = pd.read_csv(DATA_DIR / "A3_spambase.csv")

y_true = a3["y_true"].to_numpy()
score = a3["score"].to_numpy()

table = threshold_metrics_table(y_true, score)

# TODO: FPR <= 0.02, среди допустимых строк максимальный Recall.
# YOUR CODE HERE


In [ ]:
# TODO: FPR <= 0.05.
# YOUR CODE HERE


**Вывод A3:**  
Объясните смысл изменения Recall и числа ложных срабатываний.


---

# B. ROC, PR и качество ранжирования


## B1. MAGIC Gamma Telescope

**Положительный класс:** gamma event — сигнал.  
**Отрицательный класс:** hadron — фон.

Телескоп регистрирует большое число событий, среди которых нужно выделить события, похожие на интересующий исследователя gamma-сигнал.

Если фоновое событие ошибочно принять за сигнал, оно загрязняет отобранную выборку. Поэтому сначала ограничиваем долю фоновых событий, которые могут пройти через фильтр:

$$
FPR \le 0.05.
$$

После того как ограничение на фон выполнено, хочется сохранить максимальную долю настоящего сигнала. На ROC это означает: среди допустимых точек выбрать максимальный

$$
TPR = Recall.
$$

### Что нужно показать

- ROC-кривую;
- выбранные FPR, TPR и порог;
- выбранную точку на ROC;
- Precision в этой точке;
- объяснение предметного смысла FPR и TPR: какая доля фона прошла как сигнал и какая доля настоящих gamma events была сохранена.

Источник данных: UCI MAGIC Gamma Telescope, DOI `10.24432/C52C8B`.

In [ ]:
b1 = pd.read_csv(DATA_DIR / "B1_magic_gamma.csv")

y_true = b1["y_true"].to_numpy()
score = b1["score"].to_numpy()

fpr, tpr, thresholds = roc_curve(y_true, score)

# Все три массива соответствуют одним и тем же точкам ROC.
roc_points = pd.DataFrame({
    "threshold": thresholds,
    "FPR": fpr,
    "TPR": tpr,
})

roc_points.head()


In [ ]:
# TODO:
# 1. Оставьте FPR <= 0.05.
# 2. Выберите максимальный TPR.
# 3. Сохраните найденный порог.

# YOUR CODE HERE


In [ ]:
# TODO:
# Постройте ROC и отметьте найденную рабочую точку.

# YOUR CODE HERE


**Вывод B1:**  
Расшифруйте выбранные FPR и TPR словами применительно к сигналу и фону.


## B2. HTRU2: короткий список кандидатов в пульсары

**Положительный класс:** настоящий пульсар.

Автоматическая система генерирует большое число кандидатов, но настоящие пульсары среди них редки. После автоматического ранжирования наиболее интересные объекты всё равно должен проверить человек.

На отдельной выборке даны оценки **двух моделей**. Представим, что за доступное время астроном может вручную изучить только **100 кандидатов**.

Здесь недостаточно спросить, насколько хорошо модель классифицирует весь набор. Практический результат — это именно **верхняя часть ранжированного списка**. Хорошая для этой задачи модель должна поместить как можно больше настоящих пульсаров в первые 100 позиций.

### Что нужно сделать

1. построить ROC для двух моделей и сравнить ROC-AUC;
2. построить PR и сравнить Average Precision;
3. для каждой модели взять 100 объектов с максимальной оценкой;
4. сравнить Precision и Recall этих двух списков;
5. выбрать модель для ручной проверки.

### В выводе

Объясните:

- почему Accuracy на всех объектах плохо описывает качество списка из 100 кандидатов;
- помогают ли ROC-AUC и Average Precision сделать тот же выбор, что и непосредственная проверка top-100;
- какой результат оказался решающим именно для работы астронома.

Источник данных: UCI HTRU2, DOI `10.24432/C5DK6R`.

In [ ]:
b2 = pd.read_csv(DATA_DIR / "B2_htru2.csv")

y_true = b2["y_true"].to_numpy()
score_a = b2["score_a"].to_numpy()
score_b = b2["score_b"].to_numpy()

# TODO: ROC-AUC и ROC-кривые двух моделей.
# YOUR CODE HERE


In [ ]:
# TODO: AP и PR-кривые двух моделей.
# YOUR CODE HERE


In [ ]:
# TODO: сравните верхние 100 объектов.
# Подсказка: используйте top_k_metrics(...).

# YOUR CODE HERE


**Вывод B2:**  
Какую модель вы бы отдали астроному и почему?
Почему Accuracy здесь плохо описывает качество списка из 100 кандидатов?


## B3. Online Shoppers: небольшая группа для дорогого предложения

**Положительный класс:** сессия закончилась покупкой.

Представим учебный сценарий: интернет-магазин может показывать специальное предложение посетителям сайта, но это предложение дорого для компании. Поэтому показать его всем нельзя — для текущей кампании бюджет позволяет выбрать только **150 посетителей из 1000**.

После прогноза система не обязана присваивать каждому посетителю окончательный класс. Здесь нужно **ранжировать** сессии и выбрать верхнюю часть списка.

Для кампании особенно важно, сколько реальных покупок окажется среди этих 150 выбранных посетителей.

Даны оценки двух моделей.

### Что нужно сделать

- сравнить ROC-AUC и Average Precision;
- у каждой модели выбрать 150 сессий с максимальной оценкой;
- посчитать, сколько покупок оказалось в выбранной группе;
- сравнить Precision;
- выбрать модель именно для этой кампании.

Источник данных: UCI Online Shoppers Purchasing Intention, DOI `10.24432/C5F88Q`.

In [ ]:
b3 = pd.read_csv(DATA_DIR / "B3_C2_online_shoppers.csv")

y_true = b3["y_true"].to_numpy()
score_a = b3["score_a"].to_numpy()
score_b = b3["score_b"].to_numpy()

# TODO: ROC-AUC и AP двух моделей.
# YOUR CODE HERE


In [ ]:
# TODO: верхние 150 объектов у каждой модели.
# YOUR CODE HERE


**Дополнительный вопрос B3:**  
Может ли модель с немного большим ROC-AUC оказаться хуже
именно среди первых 150 объектов? Почему?


---

# C. Качество прогнозируемых вероятностей


## Небольшой мост: от вероятностей к ожидаемому числу событий

До сих пор мы чаще использовали прогноз $\hat p_i$ для **решения по отдельному объекту** или для **ранжирования** объектов.
Но иногда прикладной вопрос другой:

> **Сколько положительных событий мы ожидаем во всей группе?**

Для объекта $i$ введём индикатор события:

$$
Y_i =
\begin{cases}
1, & \text{если событие произошло},\\
0, & \text{если событие не произошло}.
\end{cases}
$$

Если $\hat p_i$ можно разумно интерпретировать как вероятность события, то

$$
\mathbb E[Y_i] \approx \hat p_i.
$$

Общее число событий в группе — это

$$
S = \sum_i Y_i.
$$

Поэтому ожидаемое число событий оценивается как

$$
\mathbb E[S]
= \sum_i \mathbb E[Y_i]
\approx \sum_i \hat p_i.
$$

То есть **сумма индивидуальных вероятностей даёт оценку ожидаемого общего числа событий**.
Для этого равенства математических ожиданий не требуется предполагать независимость объектов.

Простой пример: если для трёх объектов модель дала вероятности $0.7$, $0.2$ и $0.1$, то

$$
0.7 + 0.2 + 0.1 = 1.0.
$$

Это не означает, что в конкретной тройке обязательно произойдёт ровно одно событие.
Это означает, что модель оценивает **ожидаемое число событий** в такой группе как 1.

На валидационной выборке истинный итог уже известен, поэтому мы можем проверить,
насколько разумен вероятностный масштаб модели: сравнить $\sum_i \hat p_i$ с фактическим числом событий,
а также посмотреть calibration diagram, Log loss и Brier score.


## C1. Риск дефолта: сколько дефолтов ожидать в портфеле

**Положительный класс:** дефолт в следующем месяце.

Банк анализирует портфель из **1200 клиентов**. Здесь не нужно выбрать клиентов по порогу или сформировать короткий список.
Нужен другой результат: оценка того, **сколько дефолтов можно ожидать во всей группе**.

Для каждого клиента модель выдаёт оценку вероятности дефолта $\hat p_i$.
По идее из блока выше сумма

$$
\sum_i \hat p_i
$$

служит оценкой ожидаемого общего числа дефолтов.
На нашей отдельной выборке фактическое число дефолтов уже известно, поэтому можно проверить,
насколько разумно ведёт себя вероятностная шкала модели.

Даны прогнозы двух моделей. Они могут почти одинаково ранжировать клиентов и иметь одинаковый ROC-AUC,
но при этом выдавать вероятности разного масштаба. Для оценки общего числа дефолтов это уже принципиально.

### Что нужно сделать

- сравнить ROC-AUC и Average Precision;
- построить reliability diagram для двух моделей;
- сравнить Log loss и Brier score;
- посчитать `score_a.sum()` и `score_b.sum()`;
- сравнить эти две оценки с фактическим числом дефолтов `y_true.sum()`;
- выбрать модель для оценки общего риска портфеля.

### В выводе

Ответьте на два разных вопроса:

- одинаково ли модели ранжируют клиентов по риску;
- какая модель лучше подходит, если нужно оценить **общее ожидаемое число дефолтов**.

Объясните, почему одинаковое ранжирование ещё не гарантирует одинакового качества вероятностей.

Источник данных: UCI Default of Credit Card Clients, DOI `10.24432/C55S3H`.


In [ ]:
c1 = pd.read_csv(DATA_DIR / "C1_credit_default.csv")

y_true = c1["y_true"].to_numpy()
score_a = c1["score_a"].to_numpy()
score_b = c1["score_b"].to_numpy()

# TODO: ROC-AUC и AP.
# YOUR CODE HERE


In [ ]:
# TODO: reliability diagrams.
# Рекомендуемые параметры:
# n_bins=8, strategy="quantile"

# YOUR CODE HERE


In [ ]:
# TODO: Log loss, Brier и сумма вероятностей.
# YOUR CODE HERE


**Вывод C1:**  
Какую модель вы выбрали бы для оценки общего ожидаемого числа дефолтов?
Что показывает сравнение $\sum_i \hat p_i$ с фактическим числом дефолтов?
Почему одного ROC-AUC для этого решения недостаточно?


## C2. Online Shoppers: сколько покупок ожидать

Здесь используются **те же прогнозы**, что и в B3, но меняется прикладной вопрос.

В B3 магазин мог сделать дорогое предложение только 150 посетителям, поэтому нас интересовало качество **верхней части ранжированного списка**.

Теперь предложения никому не рассылаются. Нужно оценить, **сколько покупок произойдёт среди всех 1000 сессий**, например для оперативного планирования.

Используем идею из блока выше: если `score_a` и `score_b` имеют смысл вероятностей покупки, то

$$
\sum_i \hat p_i
$$

можно интерпретировать как оценку ожидаемого общего числа покупок.

### Что нужно сделать

- сравнить ROC-AUC и Average Precision двух моделей;
- построить reliability diagram;
- сравнить Log loss и Brier score;
- сравнить суммы прогнозируемых вероятностей с фактическим числом покупок;
- выбрать модель именно для задачи оценки общего числа покупок.

Используйте уже загруженную таблицу `b3` или снова прочитайте `B3_C2_online_shoppers.csv`.


In [ ]:
c2 = pd.read_csv(DATA_DIR / "B3_C2_online_shoppers.csv")

y_true = c2["y_true"].to_numpy()
score_a = c2["score_a"].to_numpy()
score_b = c2["score_b"].to_numpy()

# TODO: проведите сравнение для задачи оценки числа покупок.
# YOUR CODE HERE


**Вывод C2:**  
Сравните решение здесь с B3.
Почему один и тот же набор прогнозов можно оценивать по-разному
в зависимости от того, что мы собираемся с ним делать?


## C3. Occupancy Detection: разные цены ошибок

**Положительный класс:** помещение занято.

В исходных данных есть несколько показаний датчиков. В подготовленном для кейса предикторе используются температура, влажность и CO2; по ним модель оценивает, находится ли кто-то в помещении.

Представим учебный сценарий, в котором этот прогноз используется системой управления зданием.

- **FN:** помещение занято, но модель сообщает, что оно пустое. Система может слишком рано перейти в экономичный режим вентиляции или климат-контроля.
- **FP:** помещение пустое, но система считает его занятым. Ресурсы расходуются зря, но в нашей постановке эта ошибка считается менее серьёзной.

Зафиксируем условные относительные стоимости:

$$
c_{FN}=5,\qquad c_{FP}=1.
$$

Тогда для каждого порога качество решения оценивается полной стоимостью ошибок:

$$
Cost = 5\cdot FN + FP.
$$

### Что нужно сделать

- построить таблицу метрик по порогам;
- добавить столбец `cost`;
- найти порог с минимальной стоимостью;
- сравнить его с `t=0.5`;
- для двух решений сравнить FP, FN, Accuracy и итоговую стоимость;
- объяснить, почему максимум Accuracy и минимум стоимости вообще не обязаны совпадать.

Источник данных: UCI Occupancy Detection, DOI `10.24432/C5X01N`.

In [ ]:
c3 = pd.read_csv(DATA_DIR / "C3_occupancy.csv")

y_true = c3["y_true"].to_numpy()
score = c3["score"].to_numpy()

table = threshold_metrics_table(y_true, score)

# TODO:
# 1. Добавьте table["cost"] = 5 * FN + FP.
# 2. Найдите строку с минимальным cost.
# 3. Отдельно посчитайте решение при t=0.5.

# YOUR CODE HERE


**Вывод C3:**  
Как изменился порог из-за неравных цен ошибок?
Что оказалось важнее: Accuracy сама по себе или заданная функция стоимости?


---

# После семинара

Если на занятии вы успели решить только 2–3 кейса,
это нормально.

Оставшиеся задачи полезно пройти самостоятельно.
Они специально меняют постановку:

- где-то нужно почти не пропускать положительные объекты;
- где-то ограничено число доступных действий;
- где-то жёстко ограничиваются ложные срабатывания;
- где-то важен порядок объектов;
- где-то важно, чтобы сами вероятности имели разумный численный смысл.

Идея одна: **метрику и порог выбирают после постановки задачи, а не до неё.**
